# Variant Analysis Notebook

This notebook goes through the variant analysis process starting from raw FASTQ files and ending with annotated VCF files and analysis.

## Getting the Raw Data

Here, I use SRA command line function `fastq-dump` to grab the raw data from NCBI's SRA. 

In [11]:
%%bash
mkdir SRR13973983_fastqs
fastq-dump --split-3 --skip-technical --clip --read-filter pass --outdir ./SRR13973983_fastqs ./SRR13973983

mkdir: cannot create directory ‘SRR13973983_fastqs’: File exists


Rejected 43282 READS because of filtering out non-biological READS
Read 4429671 spots for ./SRR13973983
Written 4429671 spots for ./SRR13973983


For this command:  
    --split-3: separates the paired reads into their left and right ends. Any unmatched reads go into a separate file.  
    --skip-technical: Returns only biological reads. Technical reads are rejected.  
    --clip: Trims any SRA tags present in the reads  
    --readids: appends the '.1' and '.2' IDs to the respective paired reads  
    --read-filer pass: only returns reads that pass filtering  
    --outdir: specifies the output directory  

The outpur directory will contain 3 files: the two paired end files (\*_1.fastq and \*_2.fastq) and the singleton file (\*.fastq)

## Preprocessing

Once we have the raw FASTQ files, it is always a good idea to run FASTQC on the data for quality control. 

In [12]:
%%bash
mkdir SRR13973983_fastqc
fastqc -o ./SRR13973983_fastqc ./SRR13973983_fastqs/*.fastq

mkdir: cannot create directory ‘SRR13973983_fastqc’: File exists


null
null


Started analysis of SRR13973983_pass.fastq


null


Approx 5% complete for SRR13973983_pass.fastq
Approx 10% complete for SRR13973983_pass.fastq
Approx 15% complete for SRR13973983_pass.fastq
Approx 20% complete for SRR13973983_pass.fastq
Approx 25% complete for SRR13973983_pass.fastq
Approx 30% complete for SRR13973983_pass.fastq
Approx 35% complete for SRR13973983_pass.fastq
Approx 40% complete for SRR13973983_pass.fastq
Approx 45% complete for SRR13973983_pass.fastq
Approx 50% complete for SRR13973983_pass.fastq
Approx 55% complete for SRR13973983_pass.fastq
Approx 60% complete for SRR13973983_pass.fastq
Approx 65% complete for SRR13973983_pass.fastq
Approx 70% complete for SRR13973983_pass.fastq
Approx 75% complete for SRR13973983_pass.fastq
Approx 80% complete for SRR13973983_pass.fastq
Approx 85% complete for SRR13973983_pass.fastq
Approx 90% complete for SRR13973983_pass.fastq
Approx 95% complete for SRR13973983_pass.fastq


Analysis complete for SRR13973983_pass.fastq


Started analysis of SRR13973983_pass_1.fastq
Approx 5% complete for SRR13973983_pass_1.fastq
Approx 10% complete for SRR13973983_pass_1.fastq
Approx 15% complete for SRR13973983_pass_1.fastq
Approx 20% complete for SRR13973983_pass_1.fastq
Approx 25% complete for SRR13973983_pass_1.fastq
Approx 30% complete for SRR13973983_pass_1.fastq
Approx 35% complete for SRR13973983_pass_1.fastq
Approx 40% complete for SRR13973983_pass_1.fastq
Approx 45% complete for SRR13973983_pass_1.fastq
Approx 50% complete for SRR13973983_pass_1.fastq
Approx 55% complete for SRR13973983_pass_1.fastq
Approx 60% complete for SRR13973983_pass_1.fastq
Approx 65% complete for SRR13973983_pass_1.fastq
Approx 70% complete for SRR13973983_pass_1.fastq
Approx 75% complete for SRR13973983_pass_1.fastq
Approx 80% complete for SRR13973983_pass_1.fastq
Approx 85% complete for SRR13973983_pass_1.fastq
Approx 90% complete for SRR13973983_pass_1.fastq
Approx 95% complete for SRR13973983_pass_1.fastq


Analysis complete for SRR13973983_pass_1.fastq


Started analysis of SRR13973983_pass_2.fastq
Approx 5% complete for SRR13973983_pass_2.fastq
Approx 10% complete for SRR13973983_pass_2.fastq
Approx 15% complete for SRR13973983_pass_2.fastq
Approx 20% complete for SRR13973983_pass_2.fastq
Approx 25% complete for SRR13973983_pass_2.fastq
Approx 30% complete for SRR13973983_pass_2.fastq
Approx 35% complete for SRR13973983_pass_2.fastq
Approx 40% complete for SRR13973983_pass_2.fastq
Approx 45% complete for SRR13973983_pass_2.fastq
Approx 50% complete for SRR13973983_pass_2.fastq
Approx 55% complete for SRR13973983_pass_2.fastq
Approx 60% complete for SRR13973983_pass_2.fastq
Approx 65% complete for SRR13973983_pass_2.fastq
Approx 70% complete for SRR13973983_pass_2.fastq
Approx 75% complete for SRR13973983_pass_2.fastq
Approx 80% complete for SRR13973983_pass_2.fastq
Approx 85% complete for SRR13973983_pass_2.fastq
Approx 90% complete for SRR13973983_pass_2.fastq
Approx 95% complete for SRR13973983_pass_2.fastq


Analysis complete for SRR13973983_pass_2.fastq


## Alignment with BWA-MEM2

Now we move to aligning the paired reads to the hg19 reference genome. Here I will be aligning it to chromosome 5 and 17 to look for variants in the two most commonly mutated genes in the this study, APC and TP53. 

I am assuming that the `ref_genomes` directory contains the pre-index chromomes. 

In [13]:
%%bash
mkdir SRR13973983_analysis
bwa-mem2 mem ./ref_genomes/chr5.fa ./SRR13973983_fastqs/SRR13973983_pass_1.fastq SRR13973983_fastqs/SRR13973983_pass_2.fastq | samtools view -T ./ref_genomes/chr5.fa -bo SRR13973983_analysis/SRR13973983_pass_chr5.bam - 
bwa-mem2 mem ./ref_genomes/chr17.fa ./SRR13973983_fastqs/SRR13973983_pass_1.fastq SRR13973983_fastqs/SRR13973983_pass_2.fastq | samtools view -T ./ref_genomes/chr17.fa -bo SRR13973983_analysis/SRR13973983_pass_chr17.bam - 

mkdir: cannot create directory ‘SRR13973983_analysis’: File exists
Looking to launch executable "/opt/miniconda3/envs/variant_analysis/bin/bwa-mem2.avx512bw", simd = .avx512bw
Launching executable "/opt/miniconda3/envs/variant_analysis/bin/bwa-mem2.avx512bw"
-----------------------------
Executing in AVX512 mode!!
-----------------------------
* SA compression enabled with xfactor: 8
* Ref file: ./ref_genomes/chr5.fa
* Entering FMI_search
* Index file found. Loading index from ./ref_genomes/chr5.fa.bwt.2bit.64
* Reference seq len for bi-index = 361830521
* sentinel-index: 154659385
* Count:
0,	1
1,	109086479
2,	180915261
3,	252744043
4,	361830521

* Reading other elements of the index from files ./ref_genomes/chr5.fa
* Index prefix: ./ref_genomes/chr5.fa
* Read 0 ALT contigs
* Done reading Index!!
* Reading reference genome..
* Binary seq file = ./ref_genomes/chr5.fa.0123
* Reference genome size: 361830520 bp
* Done reading reference genome !!

-----------------------------------------

In [14]:
%%bash
# sorting and indexing
samtools sort -o SRR13973983_analysis/SRR13973983_pass_chr5_sort.bam SRR13973983_analysis/SRR13973983_pass_chr5.bam
samtools index SRR13973983_analysis/SRR13973983_pass_chr5_sort.bam

samtools sort -o SRR13973983_analysis/SRR13973983_pass_chr17_sort.bam SRR13973983_analysis/SRR13973983_pass_chr17.bam
samtools index SRR13973983_analysis/SRR13973983_pass_chr17_sort.bam

[bam_sort_core] merging from 3 files and 1 in-memory blocks...
[bam_sort_core] merging from 3 files and 1 in-memory blocks...


## Variant Calling with VarScan2

Now we use VarScan2 to call SNPs from our sorted .bam file. 

In [16]:
%%bash
# generating mpileup files and using VarScan2 to create .vcf files
samtools mpileup -f ./ref_genomes/chr5.fa SRR13973983_analysis/SRR13973983_pass_chr5_sort.bam | java -jar VarScan.v2.3.9.jar mpileup2snp --min-var-freq 0.01 > SRR13973983_analysis/SRR13973983_pass_chr5.vcf
samtools mpileup -f ./ref_genomes/chr17.fa SRR13973983_analysis/SRR13973983_pass_chr17_sort.bam | java -jar VarScan.v2.3.9.jar mpileup2snp --min-var-freq 0.01 > SRR13973983_analysis/SRR13973983_pass_chr17.vcf

[mpileup] 1 samples in 1 input files
Only SNPs will be reported
Min coverage:	8
Min reads2:	2
Min var freq:	0.01
Min avg qual:	15
P-value thresh:	0.01


ERROR! Session/line number was not unique in database. History logging moved to new session 15


Reading input from STDIN
25998932 bases in pileup file
94145 variant positions (91234 SNP, 2911 indel)
2697 were failed by the strand-filter
88652 variant positions reported (88652 SNP, 0 indel)
[mpileup] 1 samples in 1 input files
Only SNPs will be reported
Min coverage:	8
Min reads2:	2
Min var freq:	0.01
Min avg qual:	15
P-value thresh:	0.01
Reading input from STDIN
19127383 bases in pileup file
104821 variant positions (101928 SNP, 2893 indel)
2807 were failed by the strand-filter
99215 variant positions reported (99215 SNP, 0 indel)


## VCF Annotation with SnpEff

Using SnpEff, we can annotate our new .vcf files. This gives us information such as gene names, predicted AA changes, and functional effect predictions.

In [18]:
%%bash
snpEff hg19 SRR13973983_analysis/SRR13973983_pass_chr5.vcf > SRR13973983_analysis/SRR13973983_pass_chr5_anno.vcf
snpEff hg19 SRR13973983_analysis/SRR13973983_pass_chr17.vcf > SRR13973983_analysis/SRR13973983_pass_chr17_anno.vcf